# Hindi ASR Fine-tuning: Whisper-small
### Segment-Based Pipeline: Preprocessing → Fine-tuning → Evaluation → Error Analysis
**Model:** openai/whisper-small | **Language:** Hindi | **GPU:** GTX 1650 4GB

## Cell 0: Imports and Setup

In [1]:
import os
import json
import re
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import soundfile as sf
import librosa
import torch
import unicodedata
import warnings
warnings.filterwarnings('ignore')

# Verify GPU
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Directories
BASE_DIR    = Path('hindi_asr_data')
AUDIO_DIR   = BASE_DIR / 'audio'
TRANS_DIR   = BASE_DIR / 'transcriptions'
SEG_DIR     = BASE_DIR / 'segments'          # individual audio clips
MODEL_DIR   = Path('whisper_hindi_finetuned')

for d in [AUDIO_DIR, TRANS_DIR, SEG_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice          : {DEVICE}')
print('Directories created.')

PyTorch version : 2.5.1+cu121
CUDA available  : True
GPU             : NVIDIA GeForce GTX 1650
VRAM            : 4.3 GB

Device          : cuda
Directories created.


---
## PART A: Preprocessing

### What we do and why:
1. Load CSV manifest
2. Fix broken URLs
3. Download audio + transcription files
4. **Extract each time-stamped segment as individual training example**
5. Slice audio at exact start/end timestamps
6. Normalize Hindi text
7. Filter bad segments
8. Train/validation split

In [2]:
# ── A1. Load CSV and fix URLs ──────────────────────────────────────────────
df = pd.read_csv('FT Data - data.csv')

def fix_url(url):
    """Fix broken GCS URLs to working upload_goai format."""
    return str(url).replace(
        'https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/',
        'https://storage.googleapis.com/upload_goai/'
    )

df['rec_url_fixed']           = df['rec_url_gcp'].apply(fix_url)
df['transcription_url_fixed'] = df['transcription_url_gcp'].apply(fix_url)

print(f'Total recordings : {len(df)}')
print(f'Total duration   : {df["duration"].sum()/3600:.2f} hours')
print(f'Sample fixed URL : {df["rec_url_fixed"].iloc[0]}')

Total recordings : 104
Total duration   : 21.89 hours
Sample fixed URL : https://storage.googleapis.com/upload_goai/967179/825780_audio.wav


In [3]:
# ── A2. Download audio and transcription files ─────────────────────────────
def download_file(url, dest_path, timeout=60):
    if dest_path.exists():
        return True
    try:
        r = requests.get(url, timeout=timeout, stream=True)
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception as e:
        print(f'  FAILED {url}: {e}')
        return False

failed_audio = []
failed_trans = []

print('Downloading files (skips already downloaded)...')
for _, row in tqdm(df.iterrows(), total=len(df)):
    rid = row['recording_id']
    if not download_file(row['rec_url_fixed'], AUDIO_DIR / f'{rid}_audio.wav'):
        failed_audio.append(rid)
    if not download_file(row['transcription_url_fixed'], TRANS_DIR / f'{rid}_transcription.json'):
        failed_trans.append(rid)

print(f'Failed audio      : {len(failed_audio)}')
print(f'Failed transcripts: {len(failed_trans)}')

  0%|          | 0/104 [00:00<?, ?it/s]

100%|██████████| 104/104 [00:00<00:00, 397.48it/s]

Failed audio      : 0
Failed transcripts: 0


In [4]:
# Re-download the one failed file
url  = 'https://storage.googleapis.com/upload_goai/741178/615351_audio.wav'
dest = AUDIO_DIR / '615351_audio.wav'

print('Downloading...')
r = requests.get(url, stream=True, timeout=120)
r.raise_for_status()
with open(dest, 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

size_mb = dest.stat().st_size / 1e6
print(f'Done! File size: {size_mb:.1f} MB')

Downloading...
Done! File size: 35.5 MB


In [5]:
audio, sr = librosa.load(str(AUDIO_DIR / '615351_audio.wav'), sr=16000)
print(f'Duration: {len(audio)/sr:.1f} seconds')
print('File is good!')

Duration: 403.0 seconds
File is good!


In [6]:
# Verify ALL 104 audio files are properly downloaded and loadable
import librosa
from pathlib import Path

print('Verifying all audio files...')

good    = []
broken  = []
missing = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    rid        = row['recording_id']
    audio_path = AUDIO_DIR / f'{rid}_audio.wav'

    # Check 1: File exists
    if not audio_path.exists():
        missing.append(rid)
        continue

    # Check 2: File size is not suspiciously small (less than 1MB = broken)
    size_mb = audio_path.stat().st_size / 1e6
    if size_mb < 1.0:
        broken.append(rid)
        continue

    # Check 3: File actually loads with librosa
    try:
        audio, sr = librosa.load(str(audio_path), sr=16000, duration=5)
        good.append(rid)
    except Exception as e:
        broken.append(rid)

print(f'\nResults:')
print(f'Good files    : {len(good)}')
print(f'Missing files : {len(missing)}')
print(f'Broken files  : {len(broken)}')

if missing:
    print(f'\nMissing: {missing}')
if broken:
    print(f'\nBroken : {broken}')

if len(good) == 104:
    print('\nAll 104 files verified and good!')
else:
    print(f'\nNeed to fix {len(missing)+len(broken)} files')

Verifying all audio files...


100%|██████████| 104/104 [00:01<00:00, 72.40it/s]


Results:
Good files    : 104
Missing files : 0
Broken files  : 0

All 104 files verified and good!


In [7]:
# ── A3. Hindi Text Normalization ───────────────────────────────────────────
def normalize_hindi_text(text):
    """
    Clean Hindi text for ASR training:
    - NFC Unicode normalization (standard for Devanagari)
    - Keep only Devanagari characters and spaces
    - Remove punctuation, symbols, English characters
    - Collapse multiple spaces
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[^\u0900-\u097F\s]', '', text)  # keep only Devanagari
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test
sample = 'जो टाइम था वो!! गोल्डन टाइम होता है... school'
print('Before:', sample)
print('After :', normalize_hindi_text(sample))

Before: जो टाइम था वो!! गोल्डन टाइम होता है... school
After : जो टाइम था वो गोल्डन टाइम होता है


In [8]:
# ── A4. Extract Segments and Slice Audio ───────────────────────────────────
#
# KEY INSIGHT: Each JSON has ~92 time-stamped segments.
# Each segment = one training example.
# We slice the audio at exact start/end times.
# This gives us ~9500 examples instead of just 104.
# This is the CORRECT approach for ASR fine-tuning.
#
# Why segments and not full audio?
# Whisper processes max 30 seconds at a time.
# Full audio files are 7-20 minutes long.
# Segment-level gives proper aligned audio-text pairs.

TARGET_SR   = 16000   # Whisper requires 16kHz
MAX_DURATION = 29.0   # Whisper max is 30s, keep buffer
MIN_DURATION = 1.0    # Skip very short clips
MIN_TEXT_LEN = 5      # Skip very short transcripts

all_segments = []
skipped      = 0

print('Extracting segments from all recordings...')
print('(This slices audio at exact timestamps)')

for _, row in tqdm(df.iterrows(), total=len(df)):
    rid        = row['recording_id']
    audio_path = AUDIO_DIR / f'{rid}_audio.wav'
    trans_path = TRANS_DIR / f'{rid}_transcription.json'

    # Skip if files missing
    if not audio_path.exists() or not trans_path.exists():
        skipped += 1
        continue

    # Load full audio at 16kHz
    try:
        audio, sr = librosa.load(str(audio_path), sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f'  Audio load failed {rid}: {e}')
        skipped += 1
        continue

    # Load segments from JSON
    try:
        with open(trans_path, encoding='utf-8') as f:
            segments = json.load(f)
    except Exception as e:
        print(f'  JSON load failed {rid}: {e}')
        skipped += 1
        continue

    if not isinstance(segments, list):
        skipped += 1
        continue

    # Process each segment
    for seg_idx, seg in enumerate(segments):
        start     = seg.get('start', 0)
        end       = seg.get('end', 0)
        raw_text  = seg.get('text', '')
        speaker   = seg.get('speaker_id', '')

        duration = end - start

        # Filter 1: Duration must be between 1s and 29s
        if duration < MIN_DURATION or duration > MAX_DURATION:
            skipped += 1
            continue

        # Filter 2: Clean text must be meaningful
        clean_text = normalize_hindi_text(raw_text)
        if len(clean_text) < MIN_TEXT_LEN:
            skipped += 1
            continue

        # Slice audio at exact timestamps
        start_sample = int(start * TARGET_SR)
        end_sample   = int(end   * TARGET_SR)
        audio_clip   = audio[start_sample:end_sample]

        # Save audio clip
        seg_id        = f'{rid}_seg{seg_idx:04d}'
        seg_audio_path = SEG_DIR / f'{seg_id}.wav'
        sf.write(str(seg_audio_path), audio_clip, TARGET_SR)

        all_segments.append({
            'seg_id'      : seg_id,
            'recording_id': rid,
            'user_id'     : row['user_id'],
            'speaker_id'  : speaker,
            'start'       : start,
            'end'         : end,
            'duration'    : round(duration, 2),
            'audio_path'  : str(seg_audio_path),
            'text_raw'    : raw_text,
            'text'        : clean_text,
        })

segments_df = pd.DataFrame(all_segments)
segments_df.to_csv(BASE_DIR / 'segments_manifest.csv', index=False)

print(f'\nTotal segments extracted : {len(segments_df)}')
print(f'Skipped                  : {skipped}')
print(f'Avg segment duration     : {segments_df["duration"].mean():.1f}s')
print(f'Total training audio     : {segments_df["duration"].sum()/3600:.2f} hours')
segments_df[['seg_id','duration','text']].head(5)

Extracting segments from all recordings...
(This slices audio at exact timestamps)


100%|██████████| 104/104 [00:55<00:00,  1.86it/s]


Total segments extracted : 4442
Skipped                  : 1499
Avg segment duration     : 9.3s
Total training audio     : 11.44 hours


,seg_id,duration,text
0,825780_seg0000,14.31,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...
1,825780_seg0001,14.61,अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...
2,825780_seg0002,12.81,जंगल का सफर होता है जब हम रहने के लिए गए थे ना...
3,825780_seg0003,8.10,पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...
4,825780_seg0004,14.13,हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...


In [9]:
# ── A5. Train / Validation Split ───────────────────────────────────────────
# IMPORTANT: Split by recording_id, NOT by segment
# This prevents data leakage (segments from same recording in both train and val)

from sklearn.model_selection import train_test_split

unique_recordings = segments_df['recording_id'].unique()
train_recs, val_recs = train_test_split(
    unique_recordings,
    test_size=0.1,
    random_state=42
)

train_df = segments_df[segments_df['recording_id'].isin(train_recs)].reset_index(drop=True)
val_df   = segments_df[segments_df['recording_id'].isin(val_recs)].reset_index(drop=True)

print(f'Train segments   : {len(train_df)}')
print(f'Val segments     : {len(val_df)}')
print(f'Train duration   : {train_df["duration"].sum()/3600:.2f} hours')
print(f'Val duration     : {val_df["duration"].sum()/3600:.2f} hours')
print(f'Train recordings : {len(train_recs)}')
print(f'Val recordings   : {len(val_recs)}')

# Save splits
train_df.to_csv(BASE_DIR / 'train_manifest.csv', index=False)
val_df.to_csv(BASE_DIR / 'val_manifest.csv', index=False)
print('\nPreprocessing complete. Manifests saved.')

Train segments   : 4093
Val segments     : 349
Train duration   : 10.48 hours
Val duration     : 0.96 hours
Train recordings : 93
Val recordings   : 11

Preprocessing complete. Manifests saved.


---
## PART B Step 1: Baseline Evaluation (Pretrained Whisper-small, No Fine-tuning)

In [ ]:
from datasets import load_dataset
import evaluate
from dotenv import load_dotenv
import os
load_dotenv()

wer_metric = evaluate.load('wer')


HF_TOKEN = os.environ.get('HF_TOKEN')

print('Loading FLEURS Hindi test set...')
fleurs_test = load_dataset(
    'google/fleurs',
    'hi_in',
    split='test',
    token=HF_TOKEN
)
print(f'FLEURS test samples: {len(fleurs_test)}')
print(f'Columns: {fleurs_test.column_names}')

Loading FLEURS Hindi test set...
FLEURS test samples: 418
Columns: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']


In [11]:
# ── B2. Load Pretrained Whisper-small ──────────────────────────────────────
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = 'openai/whisper-small'

print('Loading pretrained Whisper-small...')
processor      = WhisperProcessor.from_pretrained(MODEL_NAME)
baseline_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
baseline_model.eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(language='hi', task='transcribe')
print('Baseline model ready.')

Loading pretrained Whisper-small...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Baseline model ready.


In [12]:
baseline_wer = 67.52
print(f'Baseline WER (already computed): {baseline_wer}%')

Baseline WER (already computed): 67.52%


---
## PART B Step 2: Fine-tune Whisper-small

In [13]:
# ── B4. Build HuggingFace Dataset from segments ────────────────────────────
from datasets import Dataset, Audio

def make_hf_dataset(manifest_df):
    hf = Dataset.from_dict({
        'audio'   : manifest_df['audio_path'].tolist(),
        'sentence': manifest_df['text'].tolist(),
        'seg_id'  : manifest_df['seg_id'].tolist(),
    })
    hf = hf.cast_column('audio', Audio(sampling_rate=16000))
    return hf

train_hf = make_hf_dataset(train_df)
val_hf   = make_hf_dataset(val_df)

print(f'Train HF: {len(train_hf)} segments')
print(f'Val HF  : {len(val_hf)} segments')

Train HF: 4093 segments
Val HF  : 349 segments


In [14]:
# ── B5. Feature Extraction ─────────────────────────────────────────────────
from transformers import WhisperFeatureExtractor, WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_NAME, language='Hindi', task='transcribe'
)
processor_ft = WhisperProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(
        audio['array'],
        sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = tokenizer(batch['sentence']).input_ids
    return batch

print('Preprocessing train segments...')
train_processed = train_hf.map(
    prepare_dataset,
    remove_columns=train_hf.column_names,
    num_proc=1
)
print('Preprocessing val segments...')
val_processed = val_hf.map(
    prepare_dataset,
    remove_columns=val_hf.column_names,
    num_proc=1
)
print('Feature extraction done.')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Preprocessing train segments...


Map: 100%|██████████| 4093/4093 [02:00<00:00, 34.07 examples/s]


Preprocessing val segments...


Map: 100%|██████████| 349/349 [00:07<00:00, 45.15 examples/s]

Feature extraction done.


In [15]:
# ── B6. Data Collator ──────────────────────────────────────────────────────
import dataclasses
from typing import Any, Dict, List, Union

@dataclasses.dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors='pt'
        )
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor_ft)
print('Data collator ready.')

Data collator ready.


In [16]:
# ── B7. Metrics ────────────────────────────────────────────────────────────
def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = [normalize_hindi_text(p) for p in
                 tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)]
    label_str = [normalize_hindi_text(l) for l in
                 tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': round(wer, 4)}

In [17]:
# Clear GPU memory
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory freed')
print(f'Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU memory freed
Available: 4.3 GB


In [18]:
import torch
import gc
import os

gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

ft_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
ft_model.config.forced_decoder_ids = None
ft_model.config.suppress_tokens    = []
ft_model.config.use_cache          = False

# Freeze encoder
for param in ft_model.model.encoder.parameters():
    param.requires_grad = False

training_args = Seq2SeqTrainingArguments(
    output_dir                  = str(MODEL_DIR),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 16,
    learning_rate               = 1e-5,
    warmup_steps                = 100,
    max_steps                   = 1000,
    gradient_checkpointing      = True,
    fp16                        = True,
    optim                       = 'adafactor',
    evaluation_strategy         = 'steps',
    per_device_eval_batch_size  = 1,
    predict_with_generate       = True,
    generation_max_length       = 128,
    save_steps                  = 200,
    eval_steps                  = 200,
    logging_steps               = 50,
    report_to                   = ['tensorboard'],
    load_best_model_at_end      = True,
    metric_for_best_model       = 'wer',
    greater_is_better           = False,
    push_to_hub                 = False,
    dataloader_num_workers      = 0,
    # ── NEW: reduces RAM usage during evaluation ──
    eval_accumulation_steps     = 4,       # ← evaluate in small chunks
    dataloader_pin_memory       = False,   # ← don't pin memory
)

trainer = Seq2SeqTrainer(
    args            = training_args,
    model           = ft_model,
    train_dataset   = train_processed,
    eval_dataset    = val_processed,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    tokenizer       = processor_ft.feature_extractor,
)

print('Trainer ready.')
print(f'Training on {len(train_processed)} segments')

max_steps is given, it will override any value given in num_train_epochs


Trainer ready.
Training on 4093 segments


In [19]:
print('Resuming fine-tuning from checkpoint...')
trainer.train(resume_from_checkpoint=True)
trainer.save_model(str(MODEL_DIR / 'best_model'))
processor_ft.save_pretrained(str(MODEL_DIR / 'best_model'))
print('Fine-tuning complete. Model saved.')

Resuming fine-tuning from checkpoint...


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
1001it [00:35, 28.41it/s]               There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
1001it [00:37, 26.97it/s]
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


{'train_runtime': 37.3186, 'train_samples_per_second': 428.741, 'train_steps_per_second': 26.796, 'train_loss': 0.0002220579079695634, 'epoch': 3.92}
Fine-tuning complete. Model saved.


---
## PART C: Evaluate Fine-tuned Model + WER Table

In [20]:
# ── C1. Load fine-tuned model ──────────────────────────────────────────────
ft_model_eval = WhisperForConditionalGeneration.from_pretrained(
    str(MODEL_DIR / 'best_model')
).to(DEVICE)
ft_processor = WhisperProcessor.from_pretrained(str(MODEL_DIR / 'best_model'))
forced_ids_ft = ft_processor.get_decoder_prompt_ids(language='hi', task='transcribe')
print('Fine-tuned model loaded.')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Fine-tuned model loaded.


In [21]:
# ── C2. Evaluate fine-tuned model on FLEURS ────────────────────────────────
import numpy as np

def evaluate_on_fleurs(model, proc, dataset, forced_ids, label='Model'):
    refs = []
    hyps = []
    rows = []
    model.eval()
    for sample in tqdm(dataset, desc=f'Evaluating {label}'):
        audio = np.array(sample['audio']['array'], dtype=np.float32)
        ref   = normalize_hindi_text(sample['transcription'])
        if len(ref) < 2:
            continue
        # Transcribe
        inputs = proc(
            audio,
            sampling_rate=16000,
            return_tensors='pt'
        ).input_features.to(DEVICE)
        with torch.no_grad():
            ids = model.generate(
                inputs,
                forced_decoder_ids=forced_ids,
                max_new_tokens=128
            )
        hyp = normalize_hindi_text(
            proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
        )
        refs.append(ref)
        hyps.append(hyp)
        rows.append({'reference': ref, 'hypothesis': hyp})

    wer = wer_metric.compute(references=refs, predictions=hyps)
    return round(wer * 100, 2), pd.DataFrame(rows)


print('Evaluating fine-tuned model on FLEURS Hindi test...')
print('(~20-30 minutes on GTX 1650)')
finetuned_wer, finetuned_df = evaluate_on_fleurs(
    ft_model_eval, ft_processor, fleurs_test,
    forced_ids_ft, label='Fine-tuned'
)
finetuned_df.to_csv(BASE_DIR / 'finetuned_results.csv', index=False)
print(f'Fine-tuned WER: {finetuned_wer}%')

Evaluating fine-tuned model on FLEURS Hindi test...
(~20-30 minutes on GTX 1650)


Evaluating Fine-tuned: 100%|██████████| 418/418 [10:32<00:00,  1.51s/it]


Fine-tuned WER: 49.87%


In [22]:
# ── C3. WER Results Table ──────────────────────────────────────────────────
results_table = pd.DataFrame([
    {
        'Model'         : 'Whisper-small (Pretrained Baseline)',
        'Training Data' : 'None (zero-shot)',
        'Test Set'      : 'FLEURS Hindi (hi_in)',
        'WER (%)'       : baseline_wer,
    },
    {
        'Model'         : 'Whisper-small (Fine-tuned on JoshTalks)',
        'Training Data' : f'JoshTalks Hindi ({len(segments_df)} segments)',
        'Test Set'      : 'FLEURS Hindi (hi_in)',
        'WER (%)'       : finetuned_wer,
    },
])

print('=' * 70)
print('           WORD ERROR RATE RESULTS')
print('=' * 70)
print(results_table.to_string(index=False))
print('=' * 70)
improvement = ((baseline_wer - finetuned_wer) / baseline_wer) * 100
print(f'Relative WER improvement: {improvement:.1f}%')
results_table.to_csv(BASE_DIR / 'wer_results_table.csv', index=False)

           WORD ERROR RATE RESULTS
                                  Model                   Training Data             Test Set  WER (%)
    Whisper-small (Pretrained Baseline)                None (zero-shot) FLEURS Hindi (hi_in)    67.52
Whisper-small (Fine-tuned on JoshTalks) JoshTalks Hindi (4442 segments) FLEURS Hindi (hi_in)    49.87
Relative WER improvement: 26.1%


---
## PART D: Systematic Error Sampling (≥25 utterances)

**Strategy: Stratified sampling by WER severity**
- Compute per-utterance WER for all FLEURS test samples
- Divide into 3 buckets: Low (WER<0.3), Medium (0.3-0.7), High (WER>0.7)
- Sample every Nth error from each bucket proportionally
- Total = 27 samples (9 per bucket)

In [23]:
# ── D1. Per-utterance WER ──────────────────────────────────────────────────
def per_utt_wer(ref, hyp):
    if not ref.strip():
        return None
    try:
        return round(wer_metric.compute(references=[ref], predictions=[hyp]), 4)
    except:
        return 1.0

print('Computing per-utterance WER...')
finetuned_df['per_utt_wer'] = [
    per_utt_wer(r, h)
    for r, h in zip(finetuned_df['reference'], finetuned_df['hypothesis'])
]

# Keep only errors
error_df = finetuned_df[
    finetuned_df['per_utt_wer'].notna() &
    (finetuned_df['per_utt_wer'] > 0)
].copy().reset_index(drop=True)

print(f'Total FLEURS test samples : {len(finetuned_df)}')
print(f'Samples with errors       : {len(error_df)}')
print(f'Error rate                : {len(error_df)/len(finetuned_df)*100:.1f}%')

Computing per-utterance WER...
Total FLEURS test samples : 418
Samples with errors       : 418
Error rate                : 100.0%


In [24]:
# ── D2. Stratified Sampling ────────────────────────────────────────────────
def assign_severity(w):
    if w < 0.3:   return 'Low'
    elif w < 0.7: return 'Medium'
    else:         return 'High'

error_df['severity'] = error_df['per_utt_wer'].apply(assign_severity)

print('Severity distribution:')
print(error_df['severity'].value_counts())

# Sample 9 from each bucket using systematic (every Nth) sampling
sampled = []
TARGET  = 9

for sev in ['Low', 'Medium', 'High']:
    bucket = error_df[error_df['severity'] == sev].reset_index(drop=True)
    n      = min(TARGET, len(bucket))
    step   = max(1, len(bucket) // n)
    picked = bucket.iloc[::step].head(n)
    sampled.append(picked)

sampled_df = pd.concat(sampled).reset_index(drop=True)
sampled_df.to_csv(BASE_DIR / 'sampled_errors.csv', index=False)

print(f'\nTotal sampled: {len(sampled_df)}')
print(sampled_df['severity'].value_counts())

Severity distribution:
severity
Medium    318
Low        61
High       39
Name: count, dtype: int64

Total sampled: 27
severity
Low       9
Medium    9
High      9
Name: count, dtype: int64


In [25]:
# ── D3. Display Sampled Errors ─────────────────────────────────────────────
print(f'SYSTEMATICALLY SAMPLED ERRORS (Stratified by Severity)\n')
print(f'{"#":<3} {"Sev":<7} {"WER":<6} {"Reference":<45} {"Hypothesis"}')
print('-' * 110)
for i, row in sampled_df.iterrows():
    print(f'{i+1:<3} {row["severity"]:<7} {row["per_utt_wer"]:<6} '
          f'{row["reference"][:45]:<45} {row["hypothesis"][:45]}')

SYSTEMATICALLY SAMPLED ERRORS (Stratified by Severity)

#   Sev     WER    Reference                                     Hypothesis
--------------------------------------------------------------------------------------------------------------
1   Low     0.08   हर कोई समाज से जुड़ा होता है और ट्रांसपोर्ट स हर कोई समाज से जुड़ा होता है और ट्रांसपोर्ट स
2   Low     0.1875 लोग अब कंप्यूटर स्क्रीन पर संदेश लिखते हैं शा लोग अब कंप्यूटर स्क्रीन पर संदेश लिखते हैं शौ
3   Low     0.2273 किया गया अधिकांश काम सैद्धांतिक था लेकिन सैजि किया गया अदकार्श काम सहत्दानतिक था लेकिन सैजि
4   Low     0.2692 ब्लॉगिंग एक ऐसा टूल है जो सहयोग को प्रेरित कर ब्लोगिंग एक ऐसा टूल है जो सहियोंक को प्रूरीत 
5   Low     0.2941 गीज़ा में मौजूद ग्रेट पिरामिड सात अजूबा में स गीजा में मोज़। ग्रेट पिरामीट साथ अजुभा में से
6   Low     0.1724 उनकी अनुशासित रक्षा गेंदबाजी की कला और टीम के उनकी अनशासित रक्षा गेंदबाजी की कला और टीम के 
7   Low     0.1176 चांद की सतह चट्टानों और धूल से बनी है चांद की चांद की सता है चट्टानों और

---
## PART E: Error Taxonomy

**Instructions:** After running D3, open `hindi_asr_data/sampled_errors.csv`
and read through all 27 errors manually. Then fill in the taxonomy below
with REAL examples from your data. The template shows what categories
commonly appear in Hindi ASR — match them to your actual errors.

In [26]:
taxonomy = [
    {
        'category'   : 'Category 1: Phonetic Substitution of Complex Words',
        'description': 'Model substitutes similar-sounding syllables in complex Hindi words',
        'frequency'  : 'High',
        'cause'      : 'Complex Devanagari words have similar acoustic patterns. Limited training data causes confusion between similar syllables.',
        'examples'   : [
            {
                'ref' : 'किया गया अधिकांश काम सैद्धांतिक था',
                'hyp' : 'किया गया अदकार्श काम सहत्दानतिक था',
                'note': 'अधिकांश→अदकार्श, सैद्धांतिक→सहत्दानतिक — syllable-level confusion'
            },
            {
                'ref' : 'ब्लॉगिंग एक ऐसा टूल है जो सहयोग को प्रेरित करता',
                'hyp' : 'ब्लोगिंग एक ऐसा टूल है जो सहियोंक को प्रूरीत',
                'note': 'ब्लॉगिंग→ब्लोगिंग, सहयोग→सहियोंक — vowel and consonant confusion'
            },
            {
                'ref' : 'अधिकांश छोटे द्वीप स्वतंत्र राष्ट्र हैं',
                'hyp' : 'अधिकार्ष छोटे दुईप स्वतंत्र रास्ट्र हैं',
                'note': 'Multiple word substitutions in same sentence'
            },
        ]
    },
    {
        'category'   : 'Category 2: Deletion / Merging of Syllables',
        'description': 'Model drops syllables or merges them incorrectly in long words',
        'frequency'  : 'High',
        'cause'      : 'Long Hindi words have weak middle syllables acoustically. Model skips them.',
        'examples'   : [
            {
                'ref' : 'चांद की सतह चट्टानों और धूल से बनी है',
                'hyp' : 'चांद की सता है चट्टानों और धूल से बनी है',
                'note': 'सतह→सता — syllable ह dropped, extra है inserted'
            },
            {
                'ref' : 'अस्पताल में काम करते हुए लिगिन्स ने',
                'hyp' : 'असपतल में काम करते हुए लेगिंस ने',
                'note': 'अस्पताल→असपतल — vowel marker dropped'
            },
            {
                'ref' : 'उनकी अनुशासित रक्षा गेंदबाजी',
                'hyp' : 'उनकी अनशासित रक्षा गेंदबाजी',
                'note': 'अनुशासित→अनशासित — ु dropped'
            },
        ]
    },
    {
        'category'   : 'Category 3: Loanword / Transliteration Errors',
        'description': 'English loanwords transliterated differently in hypothesis vs reference',
        'frequency'  : 'Medium',
        'cause'      : 'English loanwords can be written multiple ways in Devanagari. Model learned one variant, reference uses another.',
        'examples'   : [
            {
                'ref' : 'वाइल्ड कार्ड खरीद लेना फ़ायदेमंद',
                'hyp' : 'वाइड कार्ड कहरीद लेना फायदें बन',
                'note': 'वाइल्ड→वाइड — English wild card misheard'
            },
            {
                'ref' : 'बाद में उन्हें कैम्ब्रिज में एडेनब्रुक के',
                'hyp' : 'बाद में उन्हें केमब्रेज में एडेन भ्रुक के',
                'note': 'कैम्ब्रिज→केमब्रेज — Cambridge transliterated differently'
            },
            {
                'ref' : 'ब्लॉगिंग एक ऐसा टूल है',
                'hyp' : 'ब्लोगिंग एक ऐसा टूल है',
                'note': 'ब्लॉगिंग→ब्लोगिंग — blogging vowel variant'
            },
        ]
    },
    {
        'category'   : 'Category 4: Complete Transcription Breakdown',
        'description': 'Model produces completely wrong output — bears no resemblance to reference',
        'frequency'  : 'Low',
        'cause'      : 'Combination of fast speech, rare vocabulary, and complex sentence structure causes total failure.',
        'examples'   : [
            {
                'ref' : 'उन्होंने अफवाहों को राजनीतिक बकवास और मूर्खता',
                'hyp' : 'उम्हुंने ना खुब हो को रादनी तेख बखुआ सुर्मों',
                'note': 'WER=1.25 — almost every word wrong, rare political vocabulary'
            },
            {
                'ref' : 'मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैल',
                'hyp' : 'मदयुक के अंत्में पश्चिम यूरोप ने अपनी शैली वि',
                'note': 'WER=0.70 — historical/academic vocabulary causes breakdown'
            },
            {
                'ref' : 'मस्तिष्क विकृति और व्यवहार के बीच संबंध',
                'hyp' : 'मस्तिष्प विक्रिति और व्यवार की भी चंबंद',
                'note': 'WER=0.78 — medical/scientific terms completely misrecognized'
            },
        ]
    },
    {
        'category'   : 'Category 5: Partial Word Recognition',
        'description': 'Model correctly identifies start of word but fails on ending',
        'frequency'  : 'Medium',
        'cause'      : 'Word endings in Hindi carry grammatical meaning but are acoustically weak.',
        'examples'   : [
            {
                'ref' : 'गीज़ा में मौजूद ग्रेट पिरामिड',
                'hyp' : 'गीजा में मोज़ ग्रेट पिरामीट',
                'note': 'मौजूद→मोज़, पिरामिड→पिरामीट — word endings dropped'
            },
            {
                'ref' : 'सुंदरबन दुनिया का सबसे बड़ा तटीय मैंग्रोव',
                'hyp' : 'सुन्दर बन दुनियां का सबसे बड़ा तटी मैंग्रोप',
                'note': 'मैंग्रोव→मैंग्रोप — word ending confused'
            },
            {
                'ref' : 'उनका ऊष्मीय व्यवहार पृथ्वी की बड़ी गुफाओं',
                'hyp' : 'उनका उस्मी विवार प्रत्वी की बड़ी कुफाऊंजय',
                'note': 'Multiple word endings wrong — scientific vocabulary'
            },
        ]
    },
]

---
## PART F: Proposed Fixes for Top 3 Error Types

In [27]:
proposed_fixes = {
    'Fix 1 — Category 1 (Function Word Deletion)': """
    ERROR TYPE : Model drops short Hindi function words (है, का, में etc.)

    ROOT CAUSE : These words are acoustically weak (short, unstressed).
                 The model has insufficient exposure to their context.

    FIX        : Integrate a Hindi n-gram Language Model using shallow fusion.
                 - Train a 4-gram KenLM model on Hindi Wikipedia + CC-100 corpus
                 - During decoding, combine Whisper acoustic scores with LM scores
                 - LM penalizes grammatically incomplete sequences
                 - This forces the model to include function words

    WHY NOT JUST MORE DATA: Even with more data, function words remain
    acoustically weak. The LM provides linguistic constraints that
    acoustic models alone cannot learn reliably.

    TOOLS      : pip install pyctcdecode kenlm
    """,

    'Fix 2 — Category 3 (Code-switching)': """
    ERROR TYPE : English words in Hindi speech transcribed incorrectly

    ROOT CAUSE : Training data is pure Hindi. No exposure to mixed language.

    FIX        : Two-pronged approach:
                 1. Add code-switched training data (MUCS 2021, GLUECoS Hindi-English)
                 2. Post-processing: build a Hindi-English transliteration dictionary
                    for common loanwords (school→स्कूल, time→टाइम)
                    Apply this dictionary to normalize both reference and hypothesis
                    before WER computation

    WHY NOT JUST MORE DATA: Need specifically code-switched data, not just
    more Hindi. Also needs post-processing rules for consistent evaluation.
    """,

    'Fix 3 — Category 5 (Number Normalization)': """
    ERROR TYPE : Numbers appear as digits (42) in hypothesis but words
                 (बयालीस) in reference or vice versa

    FIX        : Apply text normalization pipeline:
                 - Convert all digits to Hindi words using num2words
                 - Apply to BOTH reference and hypothesis before WER computation
                 - Also apply during training data preprocessing
                 This removes format inconsistency as a source of error

    WHY NOT JUST MORE DATA: This is a text format mismatch problem,
    not a data quantity problem. Normalization fixes it directly.

    TOOLS      : pip install num2words
    """
}

for fix_name, fix_text in proposed_fixes.items():
    print(f'\n{"-"*70}')
    print(fix_name)
    print(fix_text)


----------------------------------------------------------------------
Fix 1 — Category 1 (Function Word Deletion)

    ERROR TYPE : Model drops short Hindi function words (है, का, में etc.)

    ROOT CAUSE : These words are acoustically weak (short, unstressed).
                 The model has insufficient exposure to their context.

    FIX        : Integrate a Hindi n-gram Language Model using shallow fusion.
                 - Train a 4-gram KenLM model on Hindi Wikipedia + CC-100 corpus
                 - During decoding, combine Whisper acoustic scores with LM scores
                 - LM penalizes grammatically incomplete sequences
                 - This forces the model to include function words

    WHY NOT JUST MORE DATA: Even with more data, function words remain
    acoustically weak. The LM provides linguistic constraints that
    acoustic models alone cannot learn reliably.

    TOOLS      : pip install pyctcdecode kenlm
    

--------------------------------------------

---
## PART G: Implement Fix 3 — Number Normalization
Show before/after results on numeral-containing test samples

In [28]:
# ── G1. Install: pip install num2words ────────────────────────────────────
from num2words import num2words

def normalize_numbers_hindi(text):
    """Convert digit sequences to Hindi words."""
    def replace_num(match):
        try:
            return num2words(int(match.group()), lang='hi')
        except:
            return match.group()
    return re.sub(r'\b\d+\b', replace_num, text)

def full_normalize(text):
    text = normalize_hindi_text(text)
    text = normalize_numbers_hindi(text)
    return text

# Test
print('Number normalization examples:')
for t in ['42 छात्र थे', '2020 में हुआ', '100 रुपये']:
    print(f'  Before: {t}')
    print(f'  After : {normalize_numbers_hindi(t)}')
    print()

Number normalization examples:
  Before: 42 छात्र थे
  After : 42 छात्र थे

  Before: 2020 में हुआ
  After : 2020 में हुआ

  Before: 100 रुपये
  After : 100 रुपये



In [29]:
# Fix: Normalize Hindi vowel markers for consistent comparison
import unicodedata

def normalize_hindi_advanced(text):
    """
    Advanced Hindi normalization:
    1. NFC normalization
    2. Normalize nukta characters
    3. Normalize similar looking characters
    4. Remove extra matras
    """
    if not isinstance(text, str):
        return ''

    # NFC normalization
    text = unicodedata.normalize('NFC', text)

    # Normalize similar characters
    # ज़ → ज, फ़ → फ (nukta variations)
    text = text.replace('\u091c\u093c', '\u091c')  # ज़ → ज
    text = text.replace('\u092b\u093c', '\u092b')  # फ़ → फ
    text = text.replace('\u0921\u093c', '\u0921')  # ड़ → ड
    text = text.replace('\u0922\u093c', '\u0922')  # ढ़ → ढ

    # Keep only Devanagari and spaces
    text = re.sub(r'[^\u0900-\u097F\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply to both ref and hyp
refs_adv = [normalize_hindi_advanced(r) for r in finetuned_df['reference'].tolist()]
hyps_adv = [normalize_hindi_advanced(h) for h in finetuned_df['hypothesis'].tolist()]

wer_advanced = wer_metric.compute(references=refs_adv, predictions=hyps_adv)

print('ADVANCED NORMALIZATION FIX — BEFORE vs AFTER')
print('=' * 50)
print(f'WER Before fix : 49.87%')
print(f'WER After fix  : {wer_advanced*100:.2f}%')
print(f'Improvement    : {(0.4987 - wer_advanced)*100:.2f} percentage points')
print('=' * 50)

ADVANCED NORMALIZATION FIX — BEFORE vs AFTER
WER Before fix : 49.87%
WER After fix  : 48.77%
Improvement    : 1.10 percentage points


In [30]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path('hindi_asr_data')

final_table = pd.DataFrame([
    {
        'Model / Configuration' : 'Whisper-small Pretrained (Baseline)',
        'Training Data'         : 'None (zero-shot)',
        'Test Set'              : 'FLEURS Hindi',
        'Post-processing'       : 'Standard normalization',
        'WER (%)'               : 67.52,
    },
    {
        'Model / Configuration' : 'Whisper-small Fine-tuned',
        'Training Data'         : 'JoshTalks (4442 segments)',
        'Test Set'              : 'FLEURS Hindi',
        'Post-processing'       : 'Standard normalization',
        'WER (%)'               : 49.87,
    },
    {
        'Model / Configuration' : 'Whisper-small Fine-tuned + Advanced Norm (Fix)',
        'Training Data'         : 'JoshTalks (4442 segments)',
        'Test Set'              : 'FLEURS Hindi',
        'Post-processing'       : 'Standard + Nukta normalization',
        'WER (%)'               : 48.77,
    },
])

print('=' * 70)
print('           FINAL COMPLETE RESULTS')
print('=' * 70)
print(final_table.to_string(index=False))
print('=' * 70)
print(f'\nTotal improvement from baseline : {67.52-48.77:.2f} percentage points')
print(f'Relative improvement            : {(67.52-48.77)/67.52*100:.1f}%')

final_table.to_csv(BASE_DIR / 'final_results_table.csv', index=False)
print('\nAll results saved!')

           FINAL COMPLETE RESULTS
                         Model / Configuration             Training Data     Test Set                Post-processing  WER (%)
           Whisper-small Pretrained (Baseline)          None (zero-shot) FLEURS Hindi         Standard normalization    67.52
                      Whisper-small Fine-tuned JoshTalks (4442 segments) FLEURS Hindi         Standard normalization    49.87
Whisper-small Fine-tuned + Advanced Norm (Fix) JoshTalks (4442 segments) FLEURS Hindi Standard + Nukta normalization    48.77

Total improvement from baseline : 18.75 percentage points
Relative improvement            : 27.8%

All results saved!


In [31]:
segments_df.to_csv(BASE_DIR / 'segments_manifest.csv', index=False)
print(f"Saved segments_manifest.csv to {BASE_DIR}")

Saved segments_manifest.csv to hindi_asr_data


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path('hindi_asr_data')

# Check segments_manifest
print("segments_manifest exists:", (BASE_DIR / 'segments_manifest.csv').exists())
print("baseline_results exists:", (BASE_DIR / 'baseline_results.csv').exists())

# Save segments_manifest if segments_df is in memory
try:
    segments_df.to_csv(BASE_DIR / 'segments_manifest.csv', index=False)
    print(f"Saved segments_manifest.csv — {len(segments_df)} rows")
except:
    print("segments_df not in memory — re-run Part A cells first")

# Save baseline_results if available
try:
    baseline_df.to_csv(BASE_DIR / 'baseline_results.csv', index=False)
    print(f"Saved baseline_results.csv — {len(baseline_df)} rows")
except:
    print("baseline_df not in memory")

segments_manifest exists: True
baseline_results exists: True
segments_df not in memory — re-run Part A cells first
baseline_df not in memory


In [2]:
import os
print(os.listdir())

['FT Data - data.csv', 'hindi_asr_data', 'hindi_asr_finetune_v2.ipynb', 'whisper_hindi_finetuned']
